In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

# **Download Visual Transformers**

In [ ]:
!pip install timm

# **Import Libraries**

In [3]:
import torch
import timm
from PIL import Image
import os
import torchvision
import tarfile
from torchvision.datasets.utils import download_url
from torch.utils.data import random_split
from torchvision.datasets import ImageFolder
from torchvision.transforms import ToTensor
from torchvision import models
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as tt
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torchvision.utils import make_grid
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import torchvision.transforms as T
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
import gc
%matplotlib inline
matplotlib.rcParams['figure.facecolor'] = '#ffffff'

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

# **Ready VTs**

In [ ]:
timm_inception_model = timm.create_model('resnet18', pretrained=True, num_classes=0)

# **Ready Dataset**

In [ ]:
!cp ./gdrive/MyDrive/bracs/train.zip ./train.zip
!unzip train.zip
!rm train.zip
!rm -r ./train/1
!rm -r ./train/2
!rm -r ./train/4
!rm -r ./train/6

In [ ]:
!cp ./gdrive/MyDrive/bracs/bracs_in_test.zip ./bracs_in_test.zip
!unzip bracs_in_test.zip
!rm bracs_in_test.zip
!rm -r ./test/1
!rm -r ./test/2
!rm -r ./test/4
!rm -r ./test/6

In [9]:
data_dir = '.'
dataset = ImageFolder(data_dir+'/train', transform=ToTensor())
random_seed = 42
torch.manual_seed(random_seed)
val_size = int(len(dataset) * 0.1)
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])
len(train_ds), len(val_ds)

(26759, 2973)

In [10]:
train_dict = {}

for i, (image, label) in enumerate(train_ds):
  if not label in train_dict.keys():
    train_dict[label] = []
  train_dict[label].append(i)

In [11]:
valid_dict = {}

for i, (image, label) in enumerate(val_ds):
  if not label in valid_dict.keys():
    valid_dict[label] = []
  valid_dict[label].append(i)

In [12]:
class CustomDataset(Dataset):
    def __init__(self, ds, main_dict):
        self.ds = ds
        self.main_dict = main_dict

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        image, label = self.ds[idx][0], self.ds[idx][1]

        chosen_positive_sample = random.choice(self.main_dict[label])
        positive_image, positive_label = self.ds[chosen_positive_sample][0], self.ds[chosen_positive_sample][1]

        all_negative_samples = []
        for key in self.main_dict.keys():
          if key != label:
            all_negative_samples += self.main_dict[key]
        chosen_negative_sample = random.choice(all_negative_samples)
        negative_image, negative_label = self.ds[chosen_negative_sample][0], self.ds[chosen_negative_sample][1]

        return image, label, positive_image, positive_label, negative_image, negative_label

In [13]:
train_ds = CustomDataset(train_ds, train_dict)
val_ds = CustomDataset(val_ds, valid_dict)

In [14]:
def get_default_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')

def to_device(data, device):
    if isinstance(data, (list,tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True)

class DeviceDataLoader():
    def __init__(self, dl, device):
        self.dl = dl
        self.device = device

    def __iter__(self):
        for b in self.dl:
            yield to_device(b, self.device)

    def __len__(self):
        return len(self.dl)

In [ ]:
device = get_default_device()
device

In [16]:
batch_size = 32
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=batch_size * 2)

# **Model Codes**

In [ ]:
!git clone https://github.com/nazari02105/pytorch-metric-learning

In [ ]:
%cd ./pytorch-metric-learning/
! pip install .
%cd ..
from pytorch_metric_learning import losses

different loss functions

In [74]:
loss_type = "TripletMarginLoss"

In [75]:
if loss_type == "TripletMarginLoss":
  loss_func = losses.TripletMarginLoss()
if loss_type == "CircleLoss":
  loss_func = losses.CircleLoss()
if loss_type == "ContrastiveLoss":
  loss_func = losses.ContrastiveLoss()
if loss_type == "IntraPairVarianceLoss":
  loss_func = losses.IntraPairVarianceLoss()
if loss_type == "GeneralizedLiftedStructureLoss":
  loss_func = losses.GeneralizedLiftedStructureLoss()
if loss_type == "MarginLoss":
  loss_func = losses.MarginLoss()
if loss_type == "MultiSimilarityLoss":
  loss_func = losses.MarginLoss()

model

In [77]:
class ImageClassificationBase(nn.Module):
    def create_input_for_pml_loss_functions(self, images, labels, positive_images, p_labels, negative_images, n_labels):
        final_embeddings = torch.cat((images, positive_images, negative_images), 0)
        final_labels = torch.cat((labels, p_labels, n_labels), 0)
        batch_size = images.shape[0]
        final_indeces_tuple = [[], [], []]
        for i in range(batch_size):
            final_indeces_tuple[0].append(i)
            final_indeces_tuple[1].append(batch_size + i)
            final_indeces_tuple[2].append(batch_size * 2 + i)
        final_indeces_tuple = (torch.tensor(final_indeces_tuple[0]), torch.tensor(final_indeces_tuple[1]), torch.tensor(final_indeces_tuple[2]))
        return final_embeddings, final_labels, final_indeces_tuple

    def training_step(self, loss_func, images, labels, positive_images, p_labels, negative_images, n_labels):
        embeddings = self(images)
        positive_embeddings = self(positive_images)
        negative_embeddings = self(negative_images)
        final_embeddings, final_labels, final_indices_tuple = self.create_input_for_pml_loss_functions(embeddings, labels, positive_embeddings, p_labels, negative_embeddings, n_labels)
        loss = loss_func(final_embeddings, final_labels, final_indices_tuple)
        return loss

    def validation_step(self, loss_func, images, labels, positive_images, p_labels, negative_images, n_labels):
        embeddings = self(images)
        positive_embeddings = self(positive_images)
        negative_embeddings = self(negative_images)
        final_embeddings, final_labels, final_indices_tuple = self.create_input_for_pml_loss_functions(embeddings, labels, positive_embeddings, p_labels, negative_embeddings, n_labels)
        loss = loss_func(final_embeddings, final_labels, final_indices_tuple)
        return {'val_loss': loss.detach()}

    def validation_epoch_end(self, outputs):
        batch_losses = [x['val_loss'] for x in outputs]
        epoch_loss = torch.stack(batch_losses).mean()
        return {'val_loss': epoch_loss.item()}

    def epoch_end(self, epoch, result):
        print("Epoch [{}], train_loss: {:.4f}, val_loss: {:.4f}".format(
            epoch, result['train_loss'], result['val_loss']))

In [78]:
class Cifar10CnnModel(ImageClassificationBase):
    def __init__(self):
        super().__init__()

        self.vt = timm_inception_model
        self.embedding = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU()
        )

    def forward(self, xb):
        xb = self.vt(xb)
        embedding = self.embedding(xb)
        return embedding

In [79]:
@torch.no_grad()
def evaluate(model, val_loader, loss_func):
    model.eval()
    outputs = list()
    for images, labels, positive_images, positive_labels, negative_images, negative_labels in tqdm(val_loader):
      outputs.append(model.validation_step(loss_func, images, labels, positive_images, positive_labels, negative_images, negative_labels))
    return model.validation_epoch_end(outputs)

In [80]:
def fit(start_epoch, epochs, lr, model, train_loader, val_loader, loss_func, optimizer):
    history = []

    for epoch in range(start_epoch, epochs):
        model.train()
        train_losses = []
        for images, labels, positive_images, positive_labels, negative_images, negative_labels in tqdm(train_loader):
            loss = model.training_step(loss_func, images, labels, positive_images, positive_labels, negative_images, negative_labels)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            train_losses.append(loss.detach())

            gc.collect()

        result = evaluate(model, val_loader, loss_func)
        result['train_loss'] = torch.stack(train_losses).mean().item()
        model.epoch_end(epoch, result)
        history.append(result)

        if epoch % 2 == 0:
          save_dict = {
              'model': model.state_dict(),
              'optimizer': optimizer.state_dict(),
              'epoch': epoch + 1
          }
          torch.save(save_dict, f"./gdrive/MyDrive/ModularRetrieval/end_of_epoch{epoch+1}.pth")

    return history

In [ ]:
model = Cifar10CnnModel()
train_dl = DeviceDataLoader(train_dl, device)
val_dl = DeviceDataLoader(val_dl, device)
to_device(model, device)

In [82]:
num_epochs = 2
lr = 0.0001
opt_func = torch.optim.Adam
optimizer = opt_func(model.parameters(), lr)

<font color="yellow">
if you have a checkpoint, give the address in below cell
</font>

In [ ]:
# this is checkpoint at the end of epoch 9
!gdown -O ./checkpoint.pth 1-E3ktYvBbkzfvwozDH4MUFtR5W-Juiv0

In [83]:
CHECKPOINT_PATH = "./checkpoint.pth"

start_epoch = 0
if os.path.isfile(CHECKPOINT_PATH):
  checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
  model.load_state_dict(checkpoint['model'], strict=False)
  model.to(device)
  optimizer.load_state_dict(checkpoint['optimizer'])
  loss_func.load_state_dict(checkpoint['loss'], strict=False)
  loss_func.to(device)
  start_epoch = int(checkpoint['epoch'])

In [ ]:
if torch.cuda.is_available():
  model = model.type(torch.cuda.FloatTensor)
history = fit(start_epoch, num_epochs, lr, model, train_dl, val_dl, loss_func, optimizer)

In [ ]:
def plot_losses(history):
    train_losses = [x.get('train_loss') for x in history]
    val_losses = [x['val_loss'] for x in history]
    plt.plot(train_losses, '-bx')
    plt.plot(val_losses, '-rx')
    plt.xlabel('epoch')
    plt.ylabel('loss')
    plt.legend(['Training', 'Validation'])
    plt.title('Loss vs. No. of epochs')

plot_losses(history)